# Initalize libraries

## Import libraries

In [ ]:
# general
import sys, os
import time
from os.path import join
from os import path
from importlib import reload
from getpass import getuser
from glob import glob
from tqdm.auto import tqdm
from multiprocessing import Pool
import shutil 
import gc

# Data
import xarray as xr
import h5py
import numpy as np
import imageio
from nexusformat.nexus import *
from PIL import Image

# Plotting
import matplotlib.pyplot as plt

# skimage
import skimage.morphology

# scipy
from scipy.ndimage import gaussian_filter
from scipy import stats
import scipy
from scipy.interpolate import griddata

# pyFAI
import pyFAI
from pyFAI.azimuthalIntegrator import AzimuthalIntegrator
from pyFAI.detectors import Detector

# Self-written libraries
sys.path.append(join(os.getcwd(), "library"))
import helper_functions as helper
import mask_lib
import interactive
from interactive import cimshow
import phase_retrieval_core as PhR
import Phase_Retrieval_old_but_gold as phr_gold

plt.rcParams["figure.constrained_layout.use"] = True  # replaces plt.tight_layout


In [ ]:
# Is there a GPU?
try:
    # Cupy
    import cupy as cp
    import cupyx as cpx

    GPU = True

    print("GPU available")

    # Self-written library
    import CCI_core_cupy as cci
except:
    GPU = False
    import CCI_core as cci

    print("GPU unavailable")
parula_map = cci.parula_map()

In [ ]:
# interactive plotting
import ipywidgets

%matplotlib widget

# Auto formatting of cells
#%load_ext jupyter_black

In [ ]:
facility = "MAXIV" # Options: "SwissFEL", "MAXI"
BEAMTIMEID = 2026021708 # Proposal number
data_fname_prefix = "2602_softimax"
USER = getuser()

# Facility specific loading functions
if facility == "PETRA":
    import PETRA_MaxP04_loading as loading
elif facility == "MAXI":
    import MAXI_loading as loading
elif facility == "SwissFEL":
    from sfdata import SFDataFiles, SFScanInfo, SFProcFile
    import Swiss_FEL_Loading as loading

    # Number or jobs for analysis
    NR_JOBS = 32
elif facility == "MAXIV":
    import MAXI_loading as loading

BASEFOLDER = "/data/visitors/softimax/20250671/%d"%BEAMTIMEID
DATAFOLDER = join(BASEFOLDER,"raw")

print("Raw Datafolder is: %s"%DATAFOLDER)

# Load dictionary for keys etc
mnemonics = loading.load_mnemonics()

### Loading data

In [ ]:
def generate_filename(scan_nr: int):
    """
    Generates filename of the given scan id

    Parameter
    =========
    scan_nr : number identifier (id) of the given scan

    Output
    ======
    filename : str
        full generated filename
    ======
    author: ck 2025
    """
    return join(DATAFOLDER, f"{data_fname_prefix}_{scan_nr:04d}.h5")

def load_key(scan_id, key):
    """
    Load any kind of data specified by key (path)
    
    Parameter
    =========
    scan_id : int
        experimental identifier of scan
    key : str
        key path of nexus file tree to relevant data field
   
    Output
    ======
    data : dict
        data dictionaray on single key
    ======
    author: ck 2024
    """
    #Generate filename from scan_id
    fname = generate_filename(scan_id)
    
    # load data with basic loading function
    data = loading.load_key(fname, key)
    
    return data

def load_data(scan_id, keypath=mnemonics["measurement"], keys=None):
    """
    Load data of all specified keys from keypath

    Parameter
    =========
    scan_id : int
        experimental identifier of scan
    keypath : str
        path of nexus file tree to relevant data field
    keys : str or list of strings
        keys to load from keypath

    Output
    ======
    data : dict
        data dictionary of keys
    ======
    author: ck 2024
    """

    # Generate filename from scan_id
    fname = generate_filename(scan_id)

    # load data with basic loading function
    data = loading.load_data(fname, keypath, keys=keys)

    return data


## Loading images

In [ ]:
def get_filepath(tiffname, camera_type = "cmos"):
    """Return raw data path from save file path."""
    head, tifffile = path.split(tiffname)
    head, scandir = path.split(head)
    return path.join(DATAFOLDER, camera_type, scandir, tifffile)

def load_tiff(tifffile):
    """load single tiff image into np array"""
    return np.array(Image.open(tifffile))

def load_tiff_list(filelist, processes=None):
    """Multiprocessing loading of image files"""
    with Pool(processes=processes) as pool:
        frames = pool.map(load_tiff, filelist)
    return np.stack(frames)

def load_cmos(scanid, file_indices = None):
    # Loads all single frames
    fname = generate_filename(scanid)
    cmos_filelist = load_key(scanid,mnemonics["cmos"])
    cmos_filelist = [get_filepath(b.decode(),camera_type = "cmos") for b in cmos_filelist]

    if file_indices is not None:
        cmos_filelist = [cmos_filelist[index] for index in file_indices]

    #print("Loading %d single frames"%len(cmos_filelist))
    return load_tiff_list(cmos_filelist)

In [ ]:
def load_images(im_id: int, file_indices=None, camera_type: str = "ccd"):
    """
    Load images corresponding to a given experimental image ID.

    Parameters
    ----------
    im_id : int
        Experimental identifier of the image scan.
    file_indices : array-like or None, optional
        Indices of frames to select from the image stack.
        If None, all frames are returned.
    camera_type : {"ccd", "cmos"}, optional
        Camera type used for acquisition.

    Returns
    -------
    images : np.ndarray
        Image stack with shape (n_frames, height, width).

    Raises
    ------
    ValueError
        If an unsupported camera_type is specified.
    ======
    author: ck 2024
    """

    if camera_type == "ccd":
        meta = load_data(im_id)

        try:
            raw_name = meta["ccd"][0]
        except (KeyError, IndexError) as exc:
            raise ValueError(f"No CCD data found for im_id={im_id}") from exc

        spe_path = (
            f"{BASEFOLDER}/raw/ccd/"
            f"{str(raw_name).split('//')[1].split('.spe')[0]}-raw.spe"
        )

        frames = imageio.mimread(spe_path, memtest="5000MB")
        images = np.squeeze(np.asarray(frames))

        if file_indices is not None:
            images = images[file_indices]

        zeros = images[...,5:]<3
        if np.sum(zeros) > 1000:
            print(f"Many zeros in image frames. Verify data transfer!")
            #raise ValueError(f"Many zeros in image frames. Verify data transfer!")
        
        images = np.stack(images)

    elif camera_type == "cmos":
        images = load_cmos(im_id, file_indices)

    else:
        raise ValueError(
            f"Unsupported camera_type '{camera_type}'. "
            "Valid options are 'ccd' and 'cmos'."
        )

    return images


### Loading image procedure

In [ ]:
# Full image loading procedure
def load_processing(im_id, file_indices = None, binning = 1, crop = None):
    """
    Loads images, averaging of two individual images (scans in tango consist of two images),
    padding to square shape, Additional cropping (optional)
    """

    # Load data
    images = load_images(im_id, file_indices = file_indices)

    # Optional cropping
    if crop is not None:
        images = images[..., :crop, :crop]

    # Binning
    if binning > 1:
        images = helper.binning(images, binning)

    # Average over all images
    if images.ndim == 4:
        image = np.mean(images, axis=(0, 1))
    elif images.ndim == 3:
        image = np.mean(images, axis=(0))
    elif images.ndim == 2:
        image = images.copy()
    images = np.stack(images)
    
    return image, images

### Loading, saving fth & cdi data

In [ ]:
# Saving of log files for fth and cdi recos
def save_fth_h5():
    # Save h5
    data = {}
    data["im_id"] = im_id
    data["topo_id"] = topo_id
    data["topo_centered"] = topo_c
    data["im_centered"] = im_c
    data["holo"] = holo
    data["recon"] = recon
    data["factor"] = factor
    data["offset"] = offset
    data["center"] = center
    data["roi"] = roi
    data["prop_dist"] = prop_dist
    data["phase"] = phase
    data["mask_bs"] = mask_pixel_smooth
    data["bs_smoothing"] = bs_smoothing
    data["experimental_setup"] = experimental_setup

    filename = join(
        folder_general, "Logs", "Data_ImId_%s_RefId_%s_%s" % (im_id, topo_id, USER)
    )
    print("Now Saving: %s" % filename)
    cci.create_hdf5(data, filename)


def save_cdi_h5():
    # Save h5
    data = {}
    data["im_id"] = im_id
    data["topo_id"] = topo_id
    data["pos"] = pos
    data["neg"] = neg
    data["factor"] = factor
    data["offset"] = offset
    data["center"] = center
    data["roi"] = roi
    data["prop_dist"] = prop_dist_cdi
    data["phase"] = phase_cdi
    data["mask_bs"] = mask_bs_cdi
    data["supportmask"] = supportmask
    data["mask_pixel"] = mask_pixel
    data["p_pc"] = p_pc
    data["n_pc"] = n_pc
    data["experimental_setup"] = experimental_setup

    filename = join(
        folder_general,
        "Logs",
        "Data_ImId_%s_RefId_%s_cdi_%s" % (im_id, topo_id, USER),
    )
    print("Now Saving: %s" % filename)
    cci.create_hdf5(data, filename)
    return

## Worker which performs complete fth reconstruction process

In [ ]:
def worker(image, topo, Norm = True):
    # Centering
    shift_c = np.array(topo.shape) / 2 - center
    topo_c = cci.shift_image(topo, shift_c)
    im_c = cci.shift_image(image, shift_c)

    ## Image registration
    shift = cci.image_registration(
       im_c[roi_im_reg],
        topo_c[roi_im_reg],
     method="dipy",
    )
    print("Relative shift is: %s" % shift)

    # Correct relative drift
    if sum(abs(shift)) > 0.05:
        im_c = cci.shift_image(im_c, -shift)
    
    if Norm:
        # Get scaling factor and offset
        factor, offset = cci.dyn_factor(
            im_c * (1 - mask_pixel),
            topo_c * (1 - mask_pixel),
            method="correlation",
            verbose=False,
            plot=False,
        )
    else:
        factor = 1
        offset = 0

    # Calculate differences (magnetic) and sums (topographc) contrast holograms.
    # _c: centered, without beamstop, _b: centered, with beamstop
    diff_c = im_c / factor - topo_c - offset
    sum_c = im_c / factor + topo_c - offset

    # Reconstruct
    recon = cci.reconstruct(
        cci.propagate(diff_c, prop_dist * 1e-6, experimental_setup=experimental_setup)
        * np.exp(1j * phase)
    )

    # worker dictionary
    worker_dict = {}
    worker_dict["center"] = center
    worker_dict["topo_c"] = topo_c
    worker_dict["im_c"] = im_c
    worker_dict["recon"] = recon
    worker_dict["factor"] = factor
    worker_dict["offset"] = offset
    worker_dict["shift"] = shift
    worker_dict["diff_c"] = diff_c
    worker_dict["sum_c"] = sum_c
    worker_dict["mask_pixel_smooth"] = mask_pixel_smooth
    worker_dict["mask_pixel"] = mask_pixel

    return worker_dict

In [ ]:
# Setup phase and propagation for cdi once
phase_cdi = 0
prop_dist_cdi = 0
dx = 0
dy = 0

def phase_retrieval_old(
    pos, neg, mask_pixel, supportmask, vmin=0, Startimage=None, Startgamma=None
):
    # Prepare Input holograms
    pos2 = pos.copy()
    neg2 = neg.copy()

    mi, _ = np.percentile(pos2[pos2 != 0], [vmin, 99.9])
    pos2 = pos2 - mi
    mi, _ = np.percentile(neg2[neg2 != 0], [vmin, 99.9])
    neg2 = neg2 - mi

    pos2[pos2 < 0] = 0
    neg2[neg2 < 0] = 0
    pos2 = pos2.astype(complex)
    neg2 = neg2.astype(complex)

    bsmask_p = mask_pixel.copy()
    bsmask_p[pos2 <= 0] = 1
    bsmask_n = mask_pixel.copy()
    bsmask_n[neg2 <= 0] = 1

    # Setup start image and startgamma
    if Startimage is None:
        Startimage = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(supportmask)))
    else:
        Startimage = Startimage.copy()
    if Startgamma is None:
        Startgamma = np.ones(pos.shape) * 1e-6 * 2
        Startgamma[pos.shape[0] // 2, pos.shape[1] // 2] = 0.7
    else:
        Startgamma = Startgamma.copy()

    # Settings for phase retrieval reconstructions
    partial_coherence = True

    # Setup
    retrieved_p = np.zeros(pos2.shape, np.cdouble)
    retrieved_n = np.zeros(pos2.shape, np.cdouble)

    # Algorithms and Inital guess
    plt.rcParams["figure.dpi"] = 100
    print("CDI - larger mask")

    algorithm_list = ["mine", "mine", "mine"]
    Nit_list = [700, 50, 50]  # iterations for algorithm_list

    x = (np.sqrt(np.maximum(pos2, np.zeros(pos2.shape)))[mask_pixel == 0]).flatten()
    y = ((np.abs(Startimage))[mask_pixel == 0]).flatten()
    res = stats.linregress(x, y)
    Startimage -= res.intercept
    Startimage /= res.slope

    average_img = 30
    real_object = False  # always set to False

    if partial_coherence:
        RL_freq = 20
        RL_it = 50

        algorithm_list_pc = ["mine", "ER", "ER"]
        Nit_list_pc = [700, 50, 50]

    # Execute Phase retrieval
    start_time = time.time()
    for i in range(len(Nit_list) // 3):
        print("############ -   CDI")

        # Positive helicity - beta_mode="arctan"
        retrieved_p, Error_diff_p, Error_supp = phr_gold.PhaseRtrv_GPU(
            diffract=np.sqrt(np.maximum(pos2, np.zeros(pos2.shape))),
            mask=supportmask,
            mode=algorithm_list[3 * i],
            beta_zero=0.5,
            Nit=Nit_list[3 * i],
            beta_mode="arctan",
            plot_every=349,
            Phase=Startimage,
            seed=False,
            real_object=real_object,
            bsmask=bsmask_p,
            average_img=average_img,
            Fourier_last=True,
        )

        # Positive helicity - beta_mode="const"
        retrieved_p, Error_diff_p2, Error_supp = phr_gold.PhaseRtrv_GPU(
            diffract=np.sqrt(np.maximum(pos2, np.zeros(pos2.shape))),
            mask=supportmask,
            mode=algorithm_list[3 * i + 1],
            beta_zero=0.5,
            Nit=Nit_list[3 * i + 1],
            beta_mode="const",
            plot_every=24,
            Phase=retrieved_p,
            seed=False,
            real_object=real_object,
            bsmask=bsmask_p,
            average_img=average_img,
            Fourier_last=True,
        )

        # Negative helicity - beta_mode="arctan"
        retrieved_n, Error_diff_n2, Error_supp = phr_gold.PhaseRtrv_GPU(
            diffract=np.sqrt(np.maximum(neg2, np.zeros(neg2.shape))),
            mask=supportmask,
            mode=algorithm_list[3 * i + 2],
            beta_zero=0.5,
            Nit=Nit_list[3 * i + 2],
            beta_mode="const",
            plot_every=24,
            Phase=retrieved_p * np.sqrt(np.sum(neg2) / np.sum(pos2)),
            seed=False,
            real_object=real_object,
            bsmask=bsmask_n,
            average_img=average_img,
            Fourier_last=True,
        )

        print("--- %s seconds ---" % np.round((time.time() - start_time), 2))

        Startimage = retrieved_p.copy()

        # Partial coherence phase retrieval
        if partial_coherence:
            # CDI_PC
            print("############   -   CDI_pc")
            pos3 = (np.abs(retrieved_p) ** 2) * bsmask_p + np.maximum(
                pos2, np.zeros(pos2.shape)
            ) * (1 - bsmask_p)
            neg3 = (np.abs(retrieved_n) ** 2) * bsmask_n + np.maximum(
                neg2, np.zeros(neg2.shape)
            ) * (1 - bsmask_n)

            # retrieve pos image
            (
                retrieved_p_pc,
                Error_diff_p_pc,
                Error_supp,
                gamma_p,
            ) = phr_gold.PhaseRtrv_with_RL(
                diffract=np.sqrt(pos3),
                mask=supportmask,
                mode=algorithm_list_pc[3 * i],
                beta_zero=0.5,
                Nit=Nit_list_pc[3 * i],
                beta_mode="arctan",
                gamma=Startgamma,
                RL_freq=RL_freq,
                RL_it=RL_it,
                plot_every=349,
                Phase=Startimage,
                seed=False,
                real_object=False,
                bsmask=np.zeros(bsmask_p.shape),
                average_img=average_img,
                Fourier_last=True,
            )

            (
                retrieved_p_pc,
                Error_diff_p_pc2,
                Error_supp,
                gamma_p,
            ) = phr_gold.PhaseRtrv_with_RL(
                diffract=np.sqrt(pos3),
                mask=supportmask,
                mode=algorithm_list[3 * i + 1],
                beta_zero=0.5,
                Nit=Nit_list_pc[3 * i + 1],
                beta_mode="const",
                gamma=gamma_p,
                RL_freq=RL_freq,
                RL_it=RL_it,
                plot_every=24,
                Phase=retrieved_p_pc,
                real_object=False,
                bsmask=np.zeros(bsmask_p.shape),
                average_img=average_img,
                Fourier_last=True,
            )
            (
                retrieved_n_pc,
                Error_diff_n_pc2,
                Error_supp,
                gamma_n,
            ) = phr_gold.PhaseRtrv_with_RL(
                diffract=np.sqrt(neg3),
                mask=supportmask,
                mode=algorithm_list[3 * i + 2],
                beta_zero=0.5,
                Nit=Nit_list_pc[3 * i + 2],
                beta_mode="const",
                gamma=gamma_p,
                RL_freq=RL_freq,
                RL_it=RL_it,
                plot_every=24,
                Phase=retrieved_p_pc * np.sqrt(np.sum(neg2) / np.sum(pos2)),
                real_object=False,
                bsmask=np.zeros(bsmask_n.shape),
                average_img=average_img,
                Fourier_last=True,
            )

            print("--- %s seconds ---" % np.round((time.time() - start_time), 2))

            Startimage = retrieved_p_pc.copy()
            Startgamma = gamma_p.copy()

    print("Phase Retrieval Done!")

    return (
        retrieved_p,
        retrieved_n,
        retrieved_p_pc,
        retrieved_n_pc,
        bsmask_p,
        bsmask_n,
        gamma_p,
        gamma_n,
    )

## Other

In [ ]:
def save_gif(output_path, image_path_list, fps=3 ):
    writer = imageio.get_writer(output_path, format="GIF-PIL", fps=fps)
    for im in tqdm(image_path_list):
        writer.append_data(imageio.imread(im))
    writer.close()

# Experimental Details

In [ ]:
# Dict with most basic experimental parameter
experimental_setup = {
    "ccd_dist": 0.13,  # ccd to sample distance
    "px_size": 20e-6,  # CMOS: 11 um, Sophia CCD: 13.5 um, Other CCD: 20µm
    "binning": 1,  # Camera binning
    "oversaturation": 2**16,  # Pixel saturation threshold
}

# Setup for azimuthal integrator
detector = Detector(
    experimental_setup["binning"] * experimental_setup["px_size"],
    experimental_setup["binning"] * experimental_setup["px_size"],
)

# General saving folder and log folder
folder_general = join(BASEFOLDER, "process")
helper.create_folder(folder_general)

print("Output Folder: %s" % folder_general)

# Load images

Start by loading the images: image of interest (im), reference of charge scattering (topo), any kind of dark image (dark)

We estalished the following convention: Difference Hologram which contains only the magnetic scattering will be calculated according to:

$Diff = \frac{Image}{factor} - Topo$,

where the factor is used for intensity scaling. In Case that you recorded scans of the same magnetic state with both helicities, use the image with negative helicity as topo and the one with positive helicity as image

In [ ]:
# Define scan id of each image or as list for multiple scans that will be averaged
im_id =  3376# single helicity mode: image with magnetic contrast, double helicity: pos+, 4416d, 4417
topo_id = 3375#single helicity mode: image without magnetic contrast, double helicity: neg

# Camera background image
dark_id_im = 3380
dark_id_topo = dark_id_im

# Optional for CMOS data
file_indices_im = None #np.arange(50) # Load specific image stack, None means all stacks
file_indices_topo = None # Load specific topo stack, None means als

# Which other meta data to load
scan_axis = "magnetOP"

# Load energy and add to experimental setup
experimental_setup["energy"] = load_key(im_id, mnemonics["energy"])
experimental_setup["lambda"] = helper.photon_energy_wavelength(
    experimental_setup["energy"], input_unit="eV"
)

print("Image Id: %s" % im_id)
print("Topo Id: %s" % topo_id)
print("Dark Id Im: %s" % dark_id_im)
print("Dark Id Topo: %s" % dark_id_topo)



## Load image of interest

In [ ]:
# Load image
if isinstance(im_id,list):
    images = np.stack([load_processing(idx)[0] for idx in im_id])
    image = np.mean(images,axis=0)
    print("list")
else:
    image, images = load_processing(im_id, file_indices = file_indices_im, crop=None)
    
# Plot
fig, ax = cimshow(helper.log_clip(image))
ax.set_title("Image")

## Load topo data set and average

In [ ]:
# Load topo
if isinstance(topo_id,list):
    topos = np.stack([load_processing(idx)[0] for idx in topo_id])
    topo = np.mean(images,axis=0)
else:
    topo, topos = load_processing(topo_id, file_indices = file_indices_topo, crop=None)

# Plot
fig, ax = cimshow(helper.log_clip(topo))
ax.set_title("Topo")

## Load dark image

In [ ]:
# Load image
if dark_id_im is not None:
    dark, darks = load_processing(dark_id_im, file_indices = None, crop=None)
    image = image - dark

    # Plot
    fig, ax = cimshow(dark)
    ax.set_title("Dark Image")

if dark_id_topo is not None:
    if dark_id_topo != dark_id_im:
        dark, _ = load_processing(dark_id_topo, crop=None) 
        
        # Plot
        fig, ax = cimshow(dark)
        ax.set_title("Dark Topo")
        
    topo = topo - dark

In [ ]:
fig, ax = cimshow(image)
ax.set_title("Image")

# Center holograms

* Find center of the hologram to get a well-defined q-space. 
* Create smooth mask for beamstop or overexposed areas in direct beam

## Basic widget to find center

Try to **align** the circles to the **center of the scattering pattern**. Care! Position of beamstop might be misleading and not represent the actual center of the hologram. Circles are just a guide to eye and will not be used otherwise.

In [ ]:
# Find center position via widget
c0, c1 = [946, 1040]  # initial values
c0, c1 = [654, 668]  # initial values
c0, c1 = [624.0, 632.5]  # initial values
ic = interactive.InteractiveCenter(topo, c0=c0, c1=c1)

In [ ]:
# Get center positions
center = [ic.c0, ic.c1]

print(f"Center:", center)

## Azimuthal integrator widget for finetuning
More of an "expert widget" which works very well for alignment if you have an Airy Pattern as a scattering image. PyFai transforms images from carthesian detector coordinate system into polar coordinate system with angle `phi` and radial distance `q` as axis (azimuthal transformation). The center of the coordinate system will be defined in the azimuthal integrator class and must not necessarily represents the center coordinates of your image array. If the center is set correctly, all rings of the Airy pattern will be transformed into a straight line in the I(q,chi)-plot as rings appear at a given q for all angles chi.

In [ ]:
# Setup azimuthal integrator for virtual geometry
ai = AzimuthalIntegrator(
    dist=experimental_setup["ccd_dist"],
    detector=detector,
    wavelength=experimental_setup["lambda"],
    poni1=center[0]
    * experimental_setup["px_size"]
    * experimental_setup["binning"],  # y (vertical)
    poni2=center[1]
    * experimental_setup["px_size"]
    * experimental_setup["binning"],  # x (horizontal)
)

In [ ]:
# Not the widget, just for double checking to find correct radial range for plotting
q_range_plotting = (0.03, 0.15)

# Perform azimuthal transformation
I_t, q_t, phi_t = ai.integrate2d(
    helper.log_clip(topo),
    500,  # number of points for phi
    radial_range=q_range_plotting,  # relevant q-range
    unit="q_nm^-1",
    correctSolidAngle=False,
    method="BBox",
)
# Combine in an xarray for plotting
az2d = xr.DataArray(I_t, dims=("phi", "q"), coords={"q": q_t, "phi": phi_t})

# Plot
fig, ax = plt.subplots()
mi, ma = np.percentile(I_t, [1, 95])
az2d.plot.imshow(ax=ax, vmin=mi, vmax=ma)
plt.title(f"Azimuthal integration")

In [ ]:
# The widget
aic = interactive.AzimuthalIntegrationCenter(
    helper.log_clip(topo),
    # image,
    ai,
    c0=center[0],
    c1=center[1],
    im_data_range=[1, 98],
    radial_range=q_range_plotting,
    qlines=[100, 110],
)

In [ ]:
# Get center positions from widget
center = [aic.c0, aic.c1]
print(f"Center:", center)

## Here: Centering of image hologram

In [ ]:
# Apply to topo and image
shift_c = np.array(image.shape) / 2 - center
im_c = cci.shift_image(image, shift_c)
topo_c = cci.shift_image(topo, shift_c)  # centered image

# Image Registration

Relative drift between data holograms and their corresponding topo holograms is calculated by image registration algorithm. Necessary to get well defined difference hologram. The reference is always the static background image (topo).

## Set Alignment ROI 

Set a region of interest (ROI) of reference (topo) use for image registration is performed. Can include beamstop when beamstop mask was defined.

How to use:
1. Zoom into the image and adjust your FOV until you are satisfied.
2. Save the axes coordinates.

In [ ]:
fig, ax = cimshow(im_c)
ax.set_title("Don't include the beamstop as this will misdirect the algorithm")

In [ ]:
# Takes start and end of x and y axis
x1, x2 = ax.get_xlim()
y2, y1 = ax.get_ylim()
roi_im_reg = np.array([y1, y2, x1, x2]).astype(int)
roi_im_reg = [469 ,694, 735, 949]
roi_im_reg_s = np.s_[roi_im_reg[0] : roi_im_reg[1], roi_im_reg[2] : roi_im_reg[3]]

print(f"Image registration roi:", roi_im_reg)

## Calculate drift of images

In [ ]:
shift = cci.image_registration(
    im_c[roi_im_reg_s],
    topo_c[roi_im_reg_s],
    method="dipy",
)
print("Relative shift is: %s" % shift)

In [ ]:
# Define shift manually for comparison
tmp_shift = [1, 2]

# Loop over shifts
temp_diff = np.zeros((3, im_c.shape[0], im_c.shape[1]))
shifts = [
    [0, 0],
    tmp_shift,
    -shift,
]
for i, tshift in enumerate(shifts):
    temp = cci.shift_image(im_c, tshift)
    temp_factor = cci.dyn_factor(temp, topo_c, method="correlation")
    temp_diff[i] = temp - temp_factor[0] * topo_c

# Plots for comparision
fig, ax = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(12, 4))
mi, ma = np.percentile(temp_diff[0], [0.1, 99.9])
ax[0].imshow(temp_diff[0], vmin=mi, vmax=ma)
ax[0].set_title("Zero shift")
mi, ma = np.percentile(temp_diff[2], [0.1, 99.9])
ax[1].imshow(temp_diff[2], vmin=mi, vmax=ma)
ax[1].set_title("Auto shift: %s px" % np.round(shifts[2], 2))
mi, ma = np.percentile(temp_diff[1], [0.1, 99.9])
ax[2].imshow(temp_diff[1], vmin=mi, vmax=ma)
ax[2].set_title("Manual shift: %s px" % shifts[1])

## Correct drift of image

In [ ]:
# Correct relative drift
if sum(abs(shift)) > 0.05:
    im_c = cci.shift_image(im_c, -shift)

# Plot original and shifted holos
mi, ma = np.percentile(np.real(topo_c[topo_c != 0]), (0.1, 99))
fig, ax = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(8, 4))
ax[0].imshow(np.real(topo), cmap="viridis", vmin=mi, vmax=ma)
ax[0].set_title("Uncentered topo")
ax[1].imshow(np.real(topo_c), cmap="viridis", vmin=mi, vmax=ma)
ax[1].set_title("Centered topo with beamstop")

# Add circles with different radi r
tmp = np.array(image.shape) / 2
for r in np.arange(50, 200, 25):
    ax[0].add_artist(plt.Circle((tmp[1], tmp[0]), r, fill=None, ec="red"))
    ax[1].add_artist(plt.Circle((tmp[1], tmp[0]), r, fill=None, ec="red"))

# Create beamstops

We want to cover the beamstop with a smooth circle to cover its sharp edges as these would create ringing-like artifacts in the reconstruction plane. Make it only as large as necessary to keep as much information as possible.

## Manual masking of beamstop wires

Just mask the beamstop wires, broken pixels, etc. 

In [ ]:
poly_mask = interactive.draw_polygon_mask(helper.log_clip(im_c))

In [ ]:
# Take poly coordinates and mask from widget
p_coord = poly_mask.get_vertice_coordinates()
mask_draw = poly_mask.full_mask.astype(int)

print("Copy these coordinates into the 'load_poly_coordinates()' function:")
print(p_coord)

# Plot image with beamstop and valid pixel mask
fig, ax = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(9, 3))
mi, ma = np.percentile(im_c * (1 - mask_draw), [0.1, 99.9])
ax[0].imshow(im_c * (1 - mask_draw), cmap="viridis", vmin=mi, vmax=ma)
ax[0].set_title("Image * (1-mask_draw)")

mi, ma = np.percentile(im_c * mask_draw, [0.1, 99.9])
ax[1].imshow(im_c * mask_draw, vmin=mi, vmax=ma)
ax[1].set_title("Image * mask_draw")

ax[2].imshow(1 - mask_draw)
ax[2].set_title("1 - mask_draw")
plt.tight_layout()

In [ ]:
def load_poly_coordinates():
    """
    Dictionary that stores polygon corner coordinates of all drawn masks
    Example: How to add masks with name "test":
    mask_coordinates["test"] = copy coordinates from above
    """
    mask_coordinates = dict()
    a = -5
    b = 2
    mask_coordinates["bs_large"] = [[(598.5+a, 609.0+b), (594.3+a, 623.9+b), (592.0+a, 643.5+b), (592.3+a, 662.6+b), (595.3+a, 672.2+b), (600.8+a, 685.6+b), (603.6+a, 693.4+b), (597.6+a, 703.2+b), (576.3+a, 726.7+b), (574.5+a, 730.4+b), (594.8+a, 743.4+b), (602.2+a, 739.7+b), (610.5+a, 728.6+b), (616.5+a, 718.4+b), (622.1+a, 714.2+b), (634.1+a, 717.0+b), (652.1+a, 722.1+b), (672.0+a, 723.0+b), (691.9+a, 717.5+b), (706.2+a, 711.5+b), (713.2+a, 716.6+b), (909.7+a, 896.9+b), (1189.2+a, 1156.2+b), (1343.9+a, 1301.5+b), (1345.6+a, 1257.2+b), (1133.8+a, 1065.6+b), (818.6+a, 773.5+b), (733.0+a, 697.4+b), (729.0+a, 692.4+b), (726.5+a, 684.5+b), (730.5+a, 677.1+b), (733.8+a, 664.7+b), (734.1+a, 650.0+b), (733.0+a, 631.1+b), (728.2+a, 617.6+b), (719.5+a, 602.9+b), (705.1+a, 589.4+b), (691.0+a, 580.3+b), (672.9+a, 576.7+b), (649.5+a, 577.5+b), (635.7+a, 582.3+b), (620.1+a, 588.8+b), (606.5+a, 577.7+b), (489.3+a, 469.5+b), (354.1+a, 345.0+b), (204.7+a, 211.5+b), (60.2+a, 75.7+b), (29.9+a, 47.3+b), (-5.9+a, 89.5+b), (-3.6+a, 128.9+b), (37.8+a, 94.0+b), (98.3+a, 149.4+b), (256.5+a, 293.8+b), (361.8+a, 389.6+b), (477.3+a, 494.9+b), (565.1+a, 575.2+b)]]
    mask_coordinates["membrane"] = [[(755.3, 643.7), (747.9, 643.7), (747.8, 649.9), (755.1, 650.0)]]
    mask_coordinates["bs_fixed"] = [[(660.7, 634.6), (656.8, 638.9), (654.0, 643.2), (653.1, 644.0), (649.4, 644.2), (648.9, 649.5), (652.6, 650.4), (653.5, 652.3), (654.7, 656.1), (657.6, 660.5), (662.2, 664.7), (665.3, 666.4), (675.6, 666.7), (680.0, 664.2), (684.3, 660.7), (686.8, 656.0), (688.6, 651.4), (687.9, 647.6), (685.9, 640.7), (681.2, 635.8), (673.3, 632.3), (664.4, 632.7)], [(602.6, 643.2), (588.7, 643.4), (589.3, 646.2), (603.5, 646.1)], [(640.0, 644.8), (622.6, 644.7), (625.4, 647.7), (633.0, 648.4), (637.7, 648.4)], [(649.1, 644.7), (580.9, 642.9), (552.7, 641.8), (511.4, 640.9), (410.2, 638.4), (410.0, 641.2), (521.7, 644.6), (599.2, 646.0), (636.9, 648.0), (649.3, 648.7), (654.2, 649.5)], [(687.3, 650.8), (725.8, 651.9), (772.2, 654.3), (844.8, 656.9), (881.4, 658.3), (938.4, 659.3), (937.5, 654.7), (833.6, 652.0), (775.4, 650.8), (720.4, 648.9), (686.6, 646.9)]]
    mask_coordinates["bs_dot"] = [[(628.5, 638.6), (627.6, 645.5), (628.8, 658.5), (631.8, 668.5), (638.5, 677.8), (646.9, 685.3), (655.0, 689.6), (665.6, 692.0), (677.1, 692.3), (688.8, 689.0), (698.2, 683.5), (707.5, 673.9), (713.8, 662.1), (714.7, 641.9), (708.4, 626.2), (699.7, 615.4), (684.3, 607.3), (670.7, 605.1), (660.5, 607.0), (647.8, 611.5), (638.5, 618.4), (633.0, 626.9)], [(639.9, 618.3), (585.4, 571.5), (533.4, 528.1), (485.3, 485.9), (431.4, 438.8), (384.4, 397.1), (336.1, 353.5), (217.6, 248.7), (145.6, 197.2), (141.4, 200.4), (170.4, 221.6), (194.5, 239.8), (223.8, 262.0), (255.2, 289.5), (318.9, 343.7), (369.8, 388.5), (422.1, 436.9), (464.7, 472.1), (507.8, 510.7), (546.5, 544.1), (571.9, 565.6), (572.8, 570.7), (578.3, 570.9), (639.2, 622.5)], [(702.1, 676.2), (718.4, 689.3), (744.2, 711.5), (763.8, 728.8), (788.8, 751.1), (833.4, 791.5), (862.3, 817.1), (903.5, 859.1), (929.4, 883.3), (970.9, 922.0), (996.1, 945.4), (1038.1, 985.0), (1040.6, 980.1), (978.3, 922.5), (930.5, 878.7), (882.8, 832.2), (843.8, 797.0), (812.0, 767.2), (761.6, 723.0), (704.0, 673.2)]]
    return mask_coordinates

In [ ]:
# Which drawn masks do you want to load? You can combine multiple masks from
# load_poly_coordinates(). Just add names of mask as strings to list like
# ["bs_small","bs_medium"]
polygon_names = ["bs_large","bs_fixed"]
mask_draw = mask_lib.load_poly_masks(
    experimental_setup["binning"] * image.shape,
    load_poly_coordinates(),
    polygon_names,
)

#mask_draw = cci.shift_image(mask_draw,[0,-8])

# Plot image with beamstop and valid pixel mask
fig, ax = plt.subplots(1, 3, sharex=True, sharey=True, figsize=(9, 3))
mi, ma = np.percentile(im_c * (1 - mask_draw), [0.1, 99.9])
ax[0].imshow(im_c * (1 - mask_draw), cmap="viridis", vmin=mi, vmax=ma)
ax[0].set_title("Image * (1-mask_draw)")

# mi, ma = np.percentile(im_c * mask_draw, [0.1, 90])
ax[1].imshow(im_c * mask_draw, vmin=mi, vmax=ma)
ax[1].set_title("Image * mask_draw")

ax[2].imshow(1 - mask_draw)
ax[2].set_title("1 - mask_draw")

## Finetuning of mask position

In [ ]:
# Use widget to shift and expand or shrink the mask
ss_mask = interactive.Shift_Scale_Mask(im_c, mask_draw, shift=[0,0], scale=0)

In [ ]:
# Take mask, shift and scaling from widget
mask_draw, mask_shift, mask_scale = ss_mask.get_mask()

## Overview beamstops
Verify good beamstop alignment

In [ ]:
# Smoothing of beamstop mask
bs_smoothing = 2

# Add circular beamstop mask
mask_im = mask_draw.copy()
mask_topo = mask_draw.copy()

# Mask over-saturated pixel
mask_im = mask_im + (im_c > experimental_setup["oversaturation"])
mask_topo = mask_topo + (topo_c > experimental_setup["oversaturation"])

# Combine both
mask_pixel = mask_im + mask_topo
mask_pixel[mask_pixel > 1] = 1

# Create smooth mask for FTH reconstructions
footprint = skimage.morphology.disk(3 * bs_smoothing)
mask_pixel_smooth = skimage.morphology.dilation(mask_pixel, footprint)
mask_pixel_smooth = gaussian_filter(mask_pixel_smooth, bs_smoothing)

# Plot both
fig, ax = plt.subplots(2, 4, figsize=(10, 5), sharex=True, sharey=True)
mi, ma = np.percentile(im_c, [1, 99.9])
ax[0, 0].imshow(im_c, vmin=mi, vmax=ma)
ax[0, 0].set_title("Image")
mi, ma = np.percentile(im_c * mask_im, [1, 99.99])
ax[0, 1].imshow(im_c * mask_im, vmin=mi, vmax=ma)
ax[0, 1].set_title("Image*mask")
mi, ma = np.percentile(im_c * (1 - mask_im), [0.1, 99.9])
ax[0, 2].imshow(im_c * (1 - mask_im), vmin=mi, vmax=ma)
ax[0, 2].set_title("Image*(1-mask)")
ax[0, 3].imshow(mask_pixel_smooth)
ax[0, 3].set_title("Combined Mask")

mi, ma = np.percentile(topo_c, [1, 99.9])
ax[1, 0].imshow(topo_c, vmin=mi, vmax=ma)
ax[1, 0].set_title("Topo")
mi, ma = np.percentile(topo_c * mask_im, [1, 99.99])
ax[1, 1].imshow(topo_c * mask_topo, vmin=mi, vmax=ma)
ax[1, 1].set_title("Topo*mask")
mi, ma = np.percentile(topo_c * (1 - mask_topo), [0.1, 99.9])
ax[1, 2].imshow(topo_c * (1 - mask_topo), vmin=mi, vmax=ma)
ax[1, 2].set_title("topo*(1-mask)")
mi, ma = np.percentile((im_c - topo_c) * (1 - mask_pixel_smooth), [0.1, 99.9])
ax[1, 3].imshow((im_c - topo_c) * (1 - mask_pixel_smooth), vmin=mi, vmax=ma)
ax[1, 3].set_title("Image-Topo")

# Here: Calculate difference holograms

You can see the reconstrution of the magnetization only after subtracting the large background that you get from the diffraction on the circular object aperture (Airy Pattern). This might require a scaling factor to correct intensity changes between the hologram and the topo. Scaling factor will be determined automatically by a linear fit. If the fit seems off, there might be an issue with the data

In [ ]:
# Get scaling factor and offset
factor, offset = cci.dyn_factor(
    im_c * (1 - mask_pixel),
    topo_c * (1 - mask_pixel),
    method="correlation",
    verbose=True,
    plot=True,
)
#factor = 1
#offset = 0
# Calculate differences (magnetic) and sums (topographc) contrast holograms.
# _c: centered, without beamstop, _b: centered, with beamstop
diff_c = im_c / factor - topo_c - offset
sum_c = im_c / factor + topo_c - offset

In [ ]:
# Plot an example of the difference or sum hologram
tmp = diff_c * (1 - mask_pixel_smooth)
# fig, ax = cimshow(np.sign(tmp) * helper.log_clip(np.abs(tmp)))
fig, ax = cimshow(tmp[243:-243,243:-243], cmap = "viridis")
ax.set_title(f" Diff Id %s - %s" % (im_id, topo_id))

#fig, ax = cimshow(sum_c* (1 - mask_pixel_smooth))
#ax.set_title(f" Sum Id %d" % im_id)

In [ ]:
tmp = np.sign(diff_c)*np.log10(np.abs(diff_c))
fig, ax = cimshow(tmp[243:-243,243:-243])

# Reconstruct Diff Holos (FTH)

Reconstruct the hologram.

0. If you are doing heraldo, determine the rotation angle of the hologram
1. Choose a region of interest (ROI) which means selecting one reconstruction from the reconstruction plane.
2. Propagate the image and shift the phase for maximal contrast and sharpness in your ROI

## Set Patterson Map ROI

Choose the reconstructions as the ROI.

1. Zoom into the image and adjust your FOV until you are satisfied.
2. Save the axes coordinates.

In [ ]:
# Choose contrast mode
# diff_c: magnetic contrast only
# sum_c: topographic contrast only
holo = diff_c * (1 - mask_pixel_smooth)
#holo = im_c * (1 - mask_pixel_smooth)
#holo = sum_c * (1 - mask_pixel_smooth)

tmp = cci.reconstruct(holo)

fig, ax = cimshow(np.real(tmp), cmap="gray")

In [ ]:
# Execute to get roi
x1, x2 = ax.get_xlim()
y2, y1 = ax.get_ylim()
roi = np.array([y1, y2, x1, x2]).astype(int)  # ystart, ystop, xstart, xstop
#roi = [ 790,  914,  885, 1004]
#roi = [ 888 , 953,  994, 1055]

roi_s = np.s_[roi[0] : roi[1], roi[2] : roi[3]]
print(f"Roi Reco:{roi}")

## Tune propagation and phase
Focus the image by tuning the propagation distance. This really works like focussing in a microscope.
Phase slider will move contrast between real and imaginary part. Usually we use the phase which maximizes the contrast in the real part.

In [ ]:
# Widget
holo = sum_c * (1 - mask_pixel_smooth)
holo = diff_c * (1 - mask_pixel_smooth)

slider_prop, slider_phase = interactive.propagate_phase(
    holo,
    roi_s,
    phase=0,  # Initial value
    prop_dist=0,  # Initial value
    experimental_setup=experimental_setup,
    scale=(.1, 99.9),
)

In [ ]:
# Read prop dist and phase from widget
prop_dist = slider_prop.value
phase = slider_phase.value

print(f"Propagation distance: %0.2f" % prop_dist)
print(f"Phase: %0.2f" % phase)

## Save reconstruction

Save png files of the images and a h5 file containing all important variables

In [ ]:
# Style of reconstruction plot
def plot_recon(recon, title, rvmin = 1, rvmax = 99):
    # Plot
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(title)

    vmin, vmax = np.percentile(np.real(recon), (rvmin, rvmax))
    t_im1 = ax[0].imshow(np.real(recon), vmin=vmin, vmax=vmax, cmap="gray")
    ax[0].set_title("Real")
    #plt.colorbar(t_im1, ax=ax[0], aspect=50)

    vmin, vmax = np.percentile(np.imag(recon), (1, 99))
    t_im2 = ax[1].imshow(np.imag(recon), vmin=vmin, vmax=vmax, cmap="gray")
    ax[1].set_title("Imag")
    #plt.colorbar(t_im2, ax=ax[1], aspect=50)

In [ ]:
def get_title(data_key, im_id, topo_id, CDI=False, Framenumber= None):
    # Rotation in title
    if data_key is not None:
        data = load_key(im_id,mnemonics[data_key])

    if CDI is False:
        mode = "FTH"
    elif CDI is True:
        mode = "CDI"

    if data_key == "magOOP":
        title = "Image %s - %s @%.3f A (OOP) - %s " % (
            im_id,
            topo_id,
            data,
            mode,
        )
    elif data_key == "magIP":
        title = "Image %s - %s @%.3f A (IP) - %s " % (
            im_id,
            topo_id,
            data,
            mode,
        )

    elif data_key == "energy":
        title = "Image %s - %s @%.3f eV - %s " % (
            im_id,
            topo_id,
            data,
            mode,
        )
    else:
        title = "Image %s - %s %s " % (
            im_id,
            topo_id,
            mode,
        )
    return title


In [ ]:
# Create plot
holo = diff_c * (1 - mask_pixel_smooth)

# Reconstruct
recon = cci.reconstruct(
        cci.propagate(holo, prop_dist * 1e-6, experimental_setup=experimental_setup)
        * np.exp(1j * phase)
    )

# Create plot
title = get_title("energy", im_id, topo_id)
plot_recon(recon[roi_s], title)

# Save images
fname = join(
    folder_general,
    "Recon_ImId_%s_RefId_%s_%s.png" % (im_id, topo_id, USER),
)
print("Saving: %s" % fname)
plt.savefig(fname, bbox_inches="tight", transparent=False)

# Save hdf5 file
#save_fth_h5()

In [ ]:
# Closes all existing plots
plt.close("all")

# Batch processing FTH (update!)

# CDI Reconstruction

## Create set of pos and neg helicity holograms

CDI algorithm needs holograms recorded wih both helicity ($\sigma = \pm 1$) as input. We use will calculate those from our previously centered and intensity normalized holograms according to;´; 

$Image(\sigma) = Topo + \sigma \cdot diff $,

In [ ]:
# Copy values from FTH reco (here topo = sum_c)
pos = (sum_c + diff_c) / 2
neg = (sum_c - diff_c) / 2

pos = im_c/factor
neg = topo_c.copy()

## Create Support mask
The support mask is the real-space constraint used for the (holographically-aided) phase retrieval, i.e., certain details about our sample like the sample geometry. For our samples we can directly derive a very strong constraint: The FTH reconstructions show us previsely the actual real-space sample structure, i.e., the arrangement of our aperture where x-rays are transmitted ("1") while the masked areas show no transmission ("0"). We will therefore create a binary mask that reflects this transmission as an input for the phase retrieval.

How to draw a support mask: Create a binary mask of the locations of sample apertures in the fth reconstruction. Areas with apertures are "1". Select only a single set of reconstructions (object & reference apertures) that originate from a single reference. Use the widget!

### Option 1: Execute if you want to create a new support mask
If you really want to create a new support mask, execute next cell and then the "InteractiveCircleCoordinates"-widget

In [ ]:
# How many references do you have?
nr_ref = 5

# Setup coordinates (nr_ref + 1 coordinates, as there is always the object aperture)
support_coordinates = [
    [pos.shape[-2] // 2, pos.shape[-1] // 2, 7] for k in range(nr_ref + 1)
]

# Widget to find the positions and sizes of the different apertures
print(
    "Cover the object & reference apertures for each set of reconstructions that originates from the same reference with circles."
)
print(
    "Optimization: Change one circle parameter, calc phase retrieval image, compare with images reconstructed with old circle parameter. Repeat!"
)

# Create plot
holo = pos * (1 - mask_pixel_smooth)

# Reconstruct
recon = cci.reconstruct(
    cci.propagate(holo, prop_dist * 1e-6, experimental_setup=experimental_setup)
    * np.exp(1j * phase)
)
recon = np.real(recon)

ds = interactive.InteractiveCircleCoordinates(
    recon,
    len(support_coordinates),
    coordinates=support_coordinates.copy(),
)

In [ ]:
# Take coordinates of circles from widget
support_coordinates = ds.get_params()

# Create supportmask from coordinates
supportmask = mask_lib.create_circle_supportmask(support_coordinates, pos.shape)

# Plot supportmask as overlay
fig, ax = plt.subplots(figsize=(6, 6))
mi, ma = np.percentile(recon, (1, 99))
ax.imshow(recon, vmin=mi, vmax=ma, cmap="gray")
ax.imshow(supportmask, alpha=0.4, cmap="binary")
ax.set_title("Image with overlayed mask")

### Option 2: Execute if you want to load an existing support mask created with circle mask widget

In [ ]:
def get_supportmask_coordinates(sample):
    """
    Dictionary that stores coordinates of circular support mask apertures
    """

    # Setup dictonary
    support_coord = dict()

    # coordinates
    support_coord["FB0022_D6"] = [(854.0, 945.0, 47.5), (922.0, 1122.0, 7.0), (779.5, 1119.0, 7.0), (681.5, 1016.5, 7.0), (1024.0, 1024.0, 7.0)]
    support_coord["s2601i_D1"] = [(920.5, 1024.5, 30.5), (952.0, 925.0, 5.0), (952.5, 1123.0, 5.0), (835.5, 1085.5, 5.0), (835.5, 963.0, 5.0), (1024.0, 1024.0, 5.0)]
    support_coord["s2601i_D1_Gd"] = [(882.0, 920.0, 46.0), (1023.0, 820.0, 5.0), (828.5, 756.5, 5.0), (708.0, 921.5, 5.0), (829.0, 1087.0, 5.0), (1024.0, 1024.0, 5.0)]
    support_coord["s2601i_D1_energy_scan"] = [(939.0, 962.0, 29.5), (1024.0, 901.6, 5.0), (906.5, 864.1, 5.0), (907.5, 1062.0, 5.0), (835.5, 963.0, 5.0), (1024.0, 1024.0, 5.0)]
    support_coord["s2602g_B3"] = [(432.0, 398.5, 87.0), (746.0, 266.5, 5.5), (404.5, 50.0, 5.5), (96.0, 314.5, 5.5), (250.0, 700.0, 5.5), (650.0, 670.0, 5.5)]
    support_coord["s2602d_C3"] = [(490.0, 470.0, 70.5), (234.0, 406.0, 9.5), (464.5, 205.5, 9.5), (722.0, 368.5, 9.5), (348.5, 693.0, 9.5), (650.0, 670.0, 9.5)]
    support_coord["s2601f_D7_cropped"] = [(174.0, 158.0, 34.5), (104.9, 266.9, 5.0), (49.0, 124.5, 6.0), (165.0, 29.0, 6.5), (294.5, 108.5, 7.0), (257.0, 256.0, 6.0)]
    support_coord["s2601f_H7_cropped"] = [(174.0, 158.0, 35.0), (104.9, 266.9, 5.0), (49.0, 124.5, 6.0), (165.0, 29.0, 6.5), (294.5, 108.5, 7.0), (257.0, 256.0, 6.0)]
    support_coord["s2601f_H7"] = [(444.0, 412.5, 85.5), (124.0, 327.0, 14.0), (418.5, 75.0, 13.5), (745.0, 286.0, 14.0), (266.0, 695.5, 13.0), (650.0, 670.0, 13.5)]
    support_coord["s2601f_J7"] = [(444.0, 408.0, 85.5), (122.0, 327.5, 14.0), (418.5, 73.5, 13.5), (744.9, 284.5, 14.0), (264.5, 696.0, 13.0), (650.0, 670.0, 13.5)]
    support_coord["s2601h_H7"] = [(440.5, 409.0, 88.5), (119.5, 329.0, 12.0), (415.5, 74.0, 12.0), (743.5, 284.0, 12.0), (264.5, 697.5, 12.0), (650.0, 670.0, 12.0)]
    support_coord["s2602g"] = [(500.5, 550.5, 49.0), (446.5, 364.5, 9.0), (656.5, 441.0, 9.0), (310.0, 546.5, 8.5), (436.5, 734.5, 10.0), (650.0, 670.0, 10.0)]
    support_coord["s2602f_J5"] = [(442.5, 548.5, 64.5), (344.5, 316.5, 11.0), (621.0, 379.5, 11.0), (390.5, 786.0, 11.0), (201.0, 568.5, 11.0), (650.0, 670.0, 11.0)]
    support_coord["s2602f_H9"] = [(444.5, 546.0, 64.5), (343.5, 317.5, 11.0), (621.0, 379.5, 11.0), (390.5, 787.0, 11.0), (201.5, 570.0, 11.0), (650.0, 670.0, 11.0)]
    return support_coord[sample]
    

In [ ]:
# Which supportmask to load? ("s2306a-C1", "s2308a-B1", ...)
sample = "s2602f_H9"

# Get coordinates and create supportmask
support_coordinates = get_supportmask_coordinates(sample)

In [ ]:
# Widget to find the positions and sizes of the different apertures
print(
    "Cover the object & reference apertures for each set of reconstructions that originates from the same reference with circles."
)
print(
    "Optimization: Change one circle parameter, calc phase retrieval image, compare with images reconstructed with old circle parameter. Repeat!"
)

# Create plot
holo = sum_c * (1 - mask_pixel_smooth)
ds = interactive.InteractiveCircleCoordinates(
    np.real(cci.reconstruct(holo)),
    len(support_coordinates),
    coordinates=support_coordinates,
)

In [ ]:
# Take coordinates of circles from widget
support_coordinates = ds.get_params()

# Create supportmask
supportmask = mask_lib.create_circle_supportmask(
    support_coordinates,pos.shape
)

# What to plot?
tmp = np.real(cci.reconstruct(holo))

# Plot supportmask as overlay
fig, ax = plt.subplots(figsize=(6, 6))
mi, ma = np.percentile(tmp, (1, 99))
ax.imshow(tmp, vmin=mi, vmax=ma, cmap="gray")
ax.imshow(supportmask, alpha=0.3, cmap="binary")
ax.set_title("Image with overlayed mask")

### Take Roi
Choose the reconstructions as the ROI.

1. Zoom into the image and adjust your FOV until you are satisfied.
2. Save the axes coordinates.

In [ ]:
fig, ax = cimshow(supportmask.astype(int))

In [ ]:
x1, x2 = ax.get_xlim()
y2, y1 = ax.get_ylim()
roi_cdi = np.array([int(y1), int(y2), int(x1), int(x2)])  # xstart, xstop, ystart, ystop
#roi_cdi = [134, 211, 119, 196]
#roi_cdi = [907, 969, 929, 993]
#roi_cdi = [813, 951, 847, 995]
#roi_cdi = [337, 524, 302, 497]
#roi_cdi = [404 ,579 ,385, 560]
#roi_cdi = [326, 543, 289, 510]
#roi_cdi = [337, 553, 296, 533]
roi_cdi = [344, 544,448, 651]
roi_cdi_s = np.s_[roi_cdi[0] : roi_cdi[1], roi_cdi[2] : roi_cdi[3]]

print("Roi:%s"%roi_cdi)

## Do Phase Retrieval

In [ ]:
# Define your recipe for the phase retrieval process. Undefined parameter are taken from default settings
phase_retrieval_recipe = dict()
phase_retrieval_recipe["hologram_intensity_cutoff_vmin"] = 1
phase_retrieval_recipe["algorithm_list_full_coherence "] = ["HAPRE","ER","ER"]
phase_retrieval_recipe["algorithm_list_partial_coherence "] = ["HAPRE","ER","ER"]
phase_retrieval_recipe["number_iterations_partial_coherence"] = [700,50,50]

In [ ]:
def smooth_supportmask(supportmask,smoothing = 0):
    if smoothing > 0:
        footprint = skimage.morphology.disk(2*smoothing)
        supportmask = skimage.morphology.erosion(supportmask, footprint)
        supportmask_smooth = gaussian_filter(supportmask,smoothing)
    else:
        supportmask_smooth = supportmask.copy()
    
    return supportmask_smooth

In [ ]:
# Executes the algorithm
(
    retrieved_p,
    retrieved_n,
    retrieved_p_pc,
    retrieved_n_pc,
    bsmask_p,
    bsmask_n,
    gamma_p,
    gamma_n,
) = PhR.phase_retrieval_algorithm(
    pos,
    neg,
    mask_pixel,
    #supportmask,
    smooth_supportmask(supportmask,smoothing = 0),
    phase_retrieval_recipe=phase_retrieval_recipe,
)

## Reconstruct images from phase retrieval

In [ ]:
# New beamstop for CDI recos as phase retrieval of low-q might be insufficient. If phase retrieval worked well
# Try without beamstop: `use_bs = False`
use_bs = False
bs_diam_cdi = 25  # diameter of beamstop

# Create beamstop
if use_bs is True:
    mask_bs_cdi = 1 - mask_lib.circle_mask(
        topo.shape, np.array(topo.shape) / 2, bs_diam_cdi, sigma=4
    )
    mask_bs_cdi = 1 - mask_pixel_smooth.copy()
elif use_bs is False:
    mask_bs_cdi = np.ones(pos.shape)  # if you don't want a beamstop

# Plotting
mode = "-"
print("Fine-tuning of reconstruction parameter:")
slider_prop, slider_phase, slider_dx, slider_dy = interactive.focusCDI(
    retrieved_p_pc * mask_bs_cdi,
    retrieved_n_pc * mask_bs_cdi,
    roi_cdi_s,
    mask=supportmask,
    phase=phase_cdi,
    dx=dx,
    dy=dy,
    prop_dist=prop_dist_cdi,
    experimental_setup=experimental_setup,
    operation=mode,
    max_prop_dist=10,
    scale=(2, 98),
)

In [ ]:
# Get phase from slider
phase_cdi = slider_phase.value
prop_dist_cdi = slider_prop.value

# Reconstruct images with new parameter
p_pc = cci.FFT(
    cci.propagate(
        retrieved_p_pc * mask_bs_cdi,
        prop_dist_cdi * 1e-6,
        experimental_setup=experimental_setup,
    )
) * np.exp(1j * phase_cdi)

n_pc = cci.FFT(
    cci.propagate(
        retrieved_n_pc * mask_bs_cdi,
        prop_dist_cdi * 1e-6,
        experimental_setup=experimental_setup,
    )
) * np.exp(1j * phase_cdi)


print("Phase CDI: %s" % phase_cdi)
print("Prop_dist: %s" % prop_dist_cdi)

In [ ]:
# Confirm that offset subtraction in cdi function works, i.e., only small fraction of hologram is actually masked
fig, ax = plt.subplots(2, 2, figsize=(8, 8), sharex=True, sharey=True)
tmp = np.abs(retrieved_p_pc * mask_bs_cdi)
mi, ma = np.percentile(tmp, [0.1, 99.9])
ax[0, 0].imshow(tmp, vmin=mi, vmax=ma)
ax[0, 0].set_title("Pos holo")

tmp = np.abs(retrieved_n_pc)
mi, ma = np.percentile(tmp, [0.1, 99.9])
ax[0, 1].imshow(tmp, vmin=mi, vmax=ma)
ax[0, 1].set_title("Neg holo")
ax[1, 0].imshow(bsmask_p)
ax[1, 0].set_title("Pos holo mask")
ax[1, 1].imshow(bsmask_n)
ax[1, 1].set_title("Neg holo mask")

In [ ]:
# cimshow(helper.log_clip(np.abs(retrieved_n_pc)))
cimshow(np.log(np.abs(retrieved_n_pc)))

In [ ]:
cimshow(np.abs(n_pc)*supportmask,cmap="gray")

## Save reconstructions

In [ ]:
def plot_recon(recon, title, scale_mask=None):
    if scale_mask is None:
        scale_mask = np.ones(recon.shape)

    # Plot
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    fig.suptitle(title)

    tmp = np.real(recon) * scale_mask
    vmin, vmax = np.nanpercentile(tmp[tmp != 0], (1, 99))
    t_im1 = ax[0].imshow(np.real(recon), vmin=vmin, vmax=vmax, cmap="gray")
    ax[0].set_title("Real")
    #plt.colorbar(t_im1, ax=ax[0], aspect=50)

    tmp = np.imag(recon) * scale_mask
    vmin, vmax = np.nanpercentile(tmp[tmp != 0], (1, 99))
    t_im2 = ax[1].imshow(np.imag(recon), vmin=vmin, vmax=vmax, cmap="gray")
    ax[1].set_title("Imag")
    #plt.colorbar(t_im2, ax=ax[1], aspect=50)

In [ ]:
cimshow(shrink_mask)

In [ ]:
cimshow(recon[roi_cdi_s]*shrink_mask[roi_cdi_s])

In [ ]:
# Saves only real and imaginary part
recon = p_pc - n_pc
#recon = (p_pc - n_pc)  / (p_pc + n_pc)
#recon = np.log(p_pc) - np.log(n_pc)


footprint = skimage.morphology.disk(1)
shrink_mask = skimage.morphology.erosion(supportmask.astype(bool), footprint)

# Plot
title = get_title("magOOP", im_id, topo_id, CDI=True, Framenumber=None)
plot_recon(
    recon[roi_cdi_s] , title, scale_mask=None#shrink_mask[roi_cdi_s]
)

# Save images
fname = join(
    folder_general,
    "Recon_ImId_%s_RefId_%s_cdi_diff_%s.png" % (im_id, topo_id, USER),
)
print("Saving: %s" % fname)
plt.savefig(fname, bbox_inches="tight", transparent=False)

# Save h5
#save_cdi_h5()

# Batch processing CDI

## Define Scan Ids

In [ ]:
# Load support mask of which sample?
sample = "s2602f_H9"

# Create supportmask from coordinates
support_coordinates = get_supportmask_coordinates(sample)
supportmask = mask_lib.create_circle_supportmask(support_coordinates, pos.shape)

In [ ]:
# Define the sets for reconstructions. You can make a list or use np.arange
# im_id_set should always have ids of positive helicity holograms,
# topo_id_set those of negative helicity or a hologram of a saturated state

# You can also use nested lists:
# in case topo_id_set = [[id1,id2],id3 it will use the sum hologram
# calculated from [id1,id2] as topo

im_id_set = np.arange(3351, 3370+1) # [896]
topo_id_set = 3351*np.ones(len(im_id_set), dtype=int)
dark_id_im_set = 3371* np.ones(len(im_id_set), dtype=int)
dark_id_topo_set = dark_id_im_set
scan_key = "magOOP"

# Use also frame_indices for indexing of single frames
use_sub_frame = False
im_frames =  None
do_cdi = True

# In case of single helicity reconstructions, adapt the helicity
helicity = [1]*len(im_id_set)

print("Dynamics Set:  %s" % im_id_set)
print("Reference Set: %s" % topo_id_set)
print("Image index Set: %s" % im_frames)

## Execute Phase Retrieval

In [ ]:
# Preprocessing which if all darks and/or topos are identical  
# Load single dark if all dark_ids are the same
if len(set(dark_id_im_set)) == 1:
    print("Loading single dark for image")
    dark_im, _ = load_processing(dark_id_im_set[0], file_indices = None, crop=None)

if len(set(topo_id_set)) == 1:
    if np.all(dark_id_im_set == dark_id_topo_set):
        dark_topo = dark_im
    else:
        print("Loading separate dark for topo")
        dark_topo, _ = load_processing(dark_id_im_set[0], file_indices = None, crop=None)      
        
    print("Loading single topo")
    topo_id = topo_id_set[0]
    topo, _ = load_processing(topo_id, file_indices = None, crop=None) 
    topo -= dark_topo

    # Center topo
    topo_c = cci.shift_image(topo, shift_c)  # centered image
    
    # Do phase retrieval on single helicity only
    # Phase retrieval of topo image only
    print("\nPerforming phase retrieval on topo only:")
    (
        retrieved_n,
        retrieved_n_pc,
        bsmask_n,
        gamma_n,
    ) = PhR.single_helicity_phase_retrieval_algorithm(
        topo_c,
        mask_pixel,
        supportmask,
        phase_retrieval_recipe=phase_retrieval_recipe,
    )

In [ ]:
# Ugly Automatic processing of image stacks
recons_name_fth, recons_name_cdi = [], []  # for gifs
images = []
corr_images= []
for it, im_id in enumerate(tqdm(im_id_set,desc="Image")):
    # Load energy and add to experimental setup
    experimental_setup["energy"] = load_key(im_id, mnemonics["energy"])
    experimental_setup["lambda"] = helper.photon_energy_wavelength(
    experimental_setup["energy"], input_unit="eV"
    )

    # Load dark
    if len(set(dark_id_im_set)) > 1:
        print("Loading dark_id: %d"%dark_id_im_set[it])
        dark_im, _ = load_processing(int(dark_id_im_set[it]), file_indices = None, crop=None)

    # load topo
    if len(set(topo_id_set)) > 1:
        topo_id = int(topo_id_set[it])
        print("Loading topo_id: %d"%topo_id)
        topo, _ = load_processing(topo_id, file_indices = None, crop=None)

        dark_topo, _ = load_processing(int(dark_id_topo_set[it]), file_indices = None, crop=None)
        topo = topo - dark_topo

    # Load images
    print("\nLoading im_id: %d"%im_id)
    image, frames = load_processing(int(im_id))
    image = image - dark_im
    frames = frames - dark_im
    corr_images.append(frames)

    # Process images
    worker_dict = worker(image, topo)

    # Reconstruct
    recon = worker_dict["recon"]
    
    # Create plot
    title = get_title("magOOP", im_id, topo_id)
    plot_recon(recon[roi_s], title)
    
    # Save images
    fname = join(
        folder_general,
        "Recon_ImId_%s_RefId_%s_stack_%s.png" % (im_id, topo_id, USER),
    )
    print("Saving: %s" % fname)
    plt.savefig(fname, bbox_inches="tight", transparent=False)

    recons_name_fth.append(fname)
    
    ################ CDI ###############
    if do_cdi == True:
        # Create pos and neg helicity set
        #pos = (worker_dict["sum_c"] + worker_dict["diff_c"]) / 2
        #neg = (worker_dict["sum_c"] - worker_dict["diff_c"]) / 2
    
        pos = worker_dict["im_c"]/worker_dict["factor"]
        neg = worker_dict["topo_c"].copy()
    
        # Create mask pixel
        mask_pixel = worker_dict["mask_pixel"]
    
        # Do fast phase retrieval if applicable
        if len(set(topo_id_set)) == 1:
            (
            retrieved_p,
            retrieved_p_pc,
            bsmask_p,
            gamma_p,
            ) = PhR.phase_retrieval_algorithm_on_second_helicity_only(
                pos,
                neg,
                retrieved_n,
                retrieved_n_pc,
                gamma_n,
                mask_pixel,
                supportmask,
                phase_retrieval_recipe=phase_retrieval_recipe)
        else:
            (
            retrieved_p,
            retrieved_n,
            retrieved_p_pc,
            retrieved_n_pc,
            bsmask_p,
            bsmask_n,
            gamma_p,
            gamma_n,
            ) = PhR.phase_retrieval_algorithm(
                pos,
                neg,
                mask_pixel,
                supportmask,
                phase_retrieval_recipe=phase_retrieval_recipe,
            )
    
        # Get Recos partial coherence
        # Positiv partial coherence
        p_pc = cci.FFT(
            cci.propagate(
                retrieved_p_pc * mask_bs_cdi,
                prop_dist_cdi * 1e-6,
                experimental_setup=experimental_setup,
            )
        )* np.exp(1j * phase_cdi)
        # Negative partial coherence
        n_pc = cci.FFT(
            cci.propagate(
                retrieved_n_pc * mask_bs_cdi,
                prop_dist_cdi * 1e-6,
                experimental_setup=experimental_setup,
            )
        )* np.exp(1j * phase_cdi)
    
        ##### Calc reco
        recon = p_pc - n_pc
        #recon = np.log(p_pc) - np.log(n_pc)
        ########
    
        # Plot
        title = get_title(scan_key, im_id, topo_id, CDI=True, Framenumber=None)
        plot_recon(
            recon[roi_cdi_s],
            title,
        )
        images.append(recon[roi_cdi_s])
        # Save images
        fname = join(
            folder_general,
            "Recon_ImId_%04d_RefId_%s_cdi_stack_%s.png" % (im_id, topo_id, USER),
        )
    
        print("Saving: %s" % fname)
        plt.savefig(fname, bbox_inches="tight", transparent=False)
        recons_name_cdi.append(fname)
    
        # Save files as h5
        #save_cdi_h5()

# Create gif
output_gif = f"FTH_{im_id_set[0]}-{im_id_set[-1]}.gif"
save_gif(join(folder_general,output_gif), recons_name_fth, fps=1)
print("CDI stack processing finished")

# Create gif
output_gif = f"CDI_{im_id_set[0]}-{im_id_set[-1]}.gif"
save_gif(join(folder_general,output_gif), recons_name_cdi, fps=1)
print("CDI stack processing finished")

In [ ]:
#quick and dirty CCI
tmp_images = np.concatenate(corr_images)
diff = np.zeros_like(tmp_images)
recos = np.zeros(tmp_images.shape,dtype="complex")

for i, im in enumerate(tmp_images):
    im  = cci.shift_image(im,shift_c)
    
    # Get scaling factor and offset
    factor, offset = cci.dyn_factor(
        im * (1 - mask_pixel),
        worker_dict["topo_c"] * (1 - mask_pixel),
        method="correlation",
        verbose=False,
        plot=False,
    )
    # Calculate differences (magnetic) and sums (topographc) contrast holograms.
    # _c: centered, without beamstop, _b: centered, with beamstop
    diff[i] = im / factor - worker_dict["topo_c"] - offset

    recos[i] = cci.reconstruct(diff[i])

In [ ]:
_ , corr_map, _ = cci.correlation_map_fast(diff*(1-mask_pixel))

fig, ax = plt.subplots()
mi, ma = np.percentile(corr_map[corr_map!=1],(1,99))
ax.imshow(corr_map,vmin=mi,vmax = ma, cmap=parula_map)
ax.invert_yaxis()

# Simon's FAAAAAST Phase Retrieval

In [ ]:
# Define the sets for reconstructions. You can make a list or use np.arange
# im_id_set should always have ids of positive helicity holograms,
# topo_id_set those of negative helicity or a hologram of a saturated state

# You can also use nested lists:
# in case topo_id_set = [[id1,id2],id3 it will use the sum hologram
# calculated from [id1,id2] as topo

im_id_set = np.arange(2878, 2882+1) # [896]
topo_id_set = [2877]*len(im_id_set)
dark_id_im_set = [2876]* np.ones(len(im_id_set), dtype=int)
dark_id_topo_set = dark_id_im_set
scan_key = "magOOP"

# Use also frame_indices for indexing of single frames
use_sub_frame = False
im_frames =  None

# In case of single helicity reconstructions, adapt the helicity
helicity = [1]*len(im_id_set)

print("Dynamics Set:  %s" % im_id_set)
print("Reference Set: %s" % topo_id_set)
print("Image index Set: %s" % im_frames)

In [ ]:
if dark_id_im_set is not None:
    
    dark_all, _ = load_processing(dark_id_im_set[0])
else:
    dark_all = np.zeros(topo.shape)
if isinstance(topo_id_set[0], list):
    print("Using Topo from sum of two helicity holograms")

    topo = (load_processing(topo_id_set[0][0])[0]+load_processing(topo_id_set[0][1])[0]) / 2

    topo_all = topo
else:
    topo_all, _ = load_processing(topo_id_set[0], file_indices=None)
normalize = False

In [ ]:
# Ugly Automatic processing of image stacks
recons_name = []  # for gifs
fth_reco_name = []
images = []
energy_list = []



for it, im_id in enumerate(tqdm(im_id_set)):
    # Load energy and add to experimental setup
    experimental_setup["energy"] = load_key(im_id, mnemonics["energy"])
    experimental_setup["lambda"] = helper.photon_energy_wavelength(
    experimental_setup["energy"], input_unit="eV"
    )
    
    # Load images
    image, _ = load_processing(im_id, file_indices=None)

    # Load dark
    if dark_id_im_set is not None:
        dark_id = dark_id_im_set[it]
        dark = dark_all.copy()#, _ = load_processing(dark_id, crop=None)
        image = image - dark

    # Get topo
    topo_id = topo_id_set[it]

    # Load data
    if isinstance(topo_id_set[0], list):
        topo_id = topo_id[0]
    else:
        print(f"Loading imageId: %04d, topoId: %04d" % (im_id, topo_id))
    #topo, _ = load_processing(topo_id)
    topo = topo_all.copy()

    if dark_id is not None:
        topo = topo - dark

    # Process images
    worker_dict = worker(image, topo, Norm=True)

    # Save topo hologram
    #save_topo_holo(worker_dict["sum_c"], im_id, topo_id)   

    
    # Reconstruct
    recon = worker_dict["recon"]
    
    # Plot
    #title = get_title("", im_id, topo_id,  Framenumber= it)
    #plot_recon(recon[roi_cdi_s], title)

   # if use_sub_frame:
        # Save images
    #    fname = join(
      #  folder_general,
     #   "Recon_ImId_%04d_RefId_%s_fth_diff_stack_%s_frame_%s.png" % (im_id, topo_id, USER, it),
   #     )
  #  else: 
        # Save images
     #   fname = join(
      #      folder_general,
     #       "Recon_ImId_%04d_RefId_%s_fth_diff_stack_%s.png" % (im_id, topo_id, USER),
        #)
    #print("Saving: %s" % fname)
    #plt.savefig(fname, bbox_inches="tight", transparent=False)
    #fth_reco_name.append(fname)
    # Optional: Save hdf5 file of fth data
    #save_fth_h5()

    ################ CDI ###############
    # Create pos and neg helicity set
    #pos = (worker_dict["sum_c"] + worker_dict["diff_c"])/2
    #neg = (worker_dict["sum_c"] - worker_dict["diff_c"])/2
    pos = worker_dict["im_c"]/worker_dict["factor"]
    neg = worker_dict["topo_c"]

    # Create mask pixel
    mask_pixel = worker_dict["mask_pixel"]

    # Create supportmask from coordinates
    supportmask = mask_lib.create_circle_supportmask(support_coordinates, pos.shape)
    offset_vmin = 0.5  
    if it == 0:
        # Executes the algorithm
        (
            retrieved_p,
            retrieved_n,
            retrieved_p_pc,
            retrieved_n_pc,
            bsmask_p,
            bsmask_n,
            gamma_p,
            gamma_n,
        ) = phase_retrieval_old(pos, neg, mask_pixel, supportmask, vmin = offset_vmin, Startimage=None, Startgamma=None)
        retrieved_p_pc_start, gamma_p_start, retrieved_n_pc_start, gamma_n_start = retrieved_p_pc, gamma_p, retrieved_n_pc, gamma_n

    elif it>0:
        # Do phase retrieval
        offset_vmin = 0.5
        mi, _ = np.percentile(pos[pos != 0], [offset_vmin, 99.9])
        pos3 = pos - mi
        (
            retrieved_p_pc,
            Error_diff_p_pc2,
            Error_supp,
            gamma_p,
        ) = phr_gold.PhaseRtrv_with_RL(
            diffract=np.sqrt(np.maximum(pos3, np.zeros(pos3.shape))),
            mask=supportmask,
            mode="ER",
            beta_zero=0.5,
            Nit=10,
            beta_mode="const",
            gamma=gamma_n_start.copy(),
            RL_freq=20,
            RL_it=50,
            Phase=retrieved_n_pc_start.copy(),
            real_object=False,
            bsmask=mask_pixel.astype(int),
            average_img=30,
            Fourier_last=True,
        )
    # Get Recos partial coherence
    # Positiv partial coherence
    p_pc = cci.FFT(
        cci.propagate(
            retrieved_p_pc * mask_bs_cdi,
            prop_dist_cdi * 1e-6,
            experimental_setup=experimental_setup,
        )
    )
    # Negative partial coherence
    n_pc = cci.FFT(
        cci.propagate(
            retrieved_n_pc * mask_bs_cdi,
            prop_dist_cdi * 1e-6,
            experimental_setup=experimental_setup,
        )
    )

    ##### Calc reco and optimze contrast
    recon = helicity[it] * (p_pc - n_pc)
    recon = recon * np.exp(1j * phase_cdi)
    print("Phase is:", np.round(phase_cdi, 2))
    ########

    # Plot
    title = get_title("magOOP", im_id, topo_id, CDI=True, Framenumber=it)
    plot_recon(recon[roi_cdi_s], title)
    images.append(recon[roi_cdi_s])
    
    if use_sub_frame:
        fname = join(
        folder_general,
        "Recon_ImId_%04d_RefId_%s_cdi_stack_%s_frame_%s.png" % (im_id, topo_id, USER, it),
        )
    else:
        fname = join(
            folder_general,
            "Recon_ImId_%04d_RefId_%s_cdi_stack_%s.png" % (im_id, topo_id, USER),
        )

    print("Saving: %s" % fname)
    plt.savefig(fname, bbox_inches="tight", transparent=False, dpi=100)
    recons_name.append(fname)

    # Save files as h5
    #save_cdi_h5()
    print(" ")

# Create gif
output_gif = f"CDI_{im_id_set[0]}-{im_id_set[-1]}.gif"
save_gif(join(folder_general,output_gif), recons_name, fps=1)
print("CDI stack processing finished")

In [ ]:
from matplotlib import cm
from PIL import Image
import os
import shutil
import matplotlib.cm as cm
from matplotlib.gridspec import GridSpec
import imageio

#plt.close("all")
def save_gif(output_path, image_path_list, fps=3 ):
    writer = imageio.get_writer(output_path, format="GIF-PIL", fps=fps)
    for im in tqdm(image_path_list):
        writer.append_data(imageio.imread(im))
    writer.close()

# Temporärer Ordner für Zwischenbilder
tmp_dir = "tmp"
os.makedirs(tmp_dir, exist_ok=True)

# Beispiel: 3D-Array mit komplexen Daten (ersetze dies durch dein tatsächliches Array)
complex_array = np.array(images)
z, y, x = complex_array.shape  # Dimensionen des Arrays

frame_paths = []

# Erstelle die GIF-Frames
for idx, i in enumerate(tqdm(range(complex_array.shape[0]))):  # Schleife über die erste Achse
    real_part = np.real(complex_array[i])  # Realteil
    imag_part = np.imag(complex_array[i])  # Imaginärteil
    
    

    fig, ax = plt.subplots(1,2, figsize=(10, 6))

    # Titel
    fig.suptitle(get_title("magOOP", im_id, topo_id, CDI=True))

    # Realteil plot (oben links)
    im1 = ax[0].imshow(real_part, cmap=cm.gray, origin='lower', 
                     vmin=np.min(complex_array.real), vmax=np.max(complex_array.real))
    ax[0].set_title("Real")
    fig.colorbar(im1, ax=ax[0], fraction=0.046, pad=0.04)

    # Imaginärteil plot (oben rechts)
    im2 = ax[1].imshow(imag_part, cmap=cm.gray, origin='lower', 
                     vmin=np.min(complex_array.imag), vmax=np.max(complex_array.imag))
    ax[1].set_title("Imaginary")
    fig.colorbar(im2, ax=ax[1], fraction=0.046, pad=0.04)
    
    # Speichere die aktuelle Figur als Bild im temporären Ordner
    frame_path = os.path.join(tmp_dir, f'frame_{i:03d}.png')
    frame_paths.append(frame_path)
    plt.savefig(frame_path, dpi=200)
    plt.close(fig)
    

# Erstelle das GIF mit create_gif
output_gif = folder_general+f"/{im_id_set[0]}-{topo_id_set[0]}_norm_{normalize}.gif"

save_gif(output_gif, frame_paths, fps=8)
#print(f"GIF gespeichert als '{gif_path}'")

# Lösche den temporären Ordner
shutil.rmtree(tmp_dir)
print(f"Temporärer Ordner '{tmp_dir}' entfernt.")


In [ ]:
plt.close("all")

## Testing Area